In [15]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# 1. Charge les variables cachées dans le fichier .env
load_dotenv()

# 2. Récupère le jeton
mon_token = os.getenv("HF_TOKEN")

# 3. Authentifie la session courante
login(token=mon_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [16]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load a tokenizer to use its chat template
template_tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct"
 )
#template_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

In [17]:
from pathlib import Path

patterns = [
    "data/1981/legislatives/*PF*.txt",
    "data/1988/legislatives/*PF*.txt",
    "data/1993/legislatives/*PF*.txt"
]

# 1. On charge d'abord tous vos fichiers dans un seul bloc global
dataset_complet = load_dataset("text", data_files=patterns, split="train", sample_by= "document")

# récupérer les fichiers dans le même ordre
files = []
for p in patterns:
    files.extend(sorted(Path().glob(p)))

def add_id(example, idx):
    path = files[idx]
    example["id"] = path.stem
    example["annee"] = path.parts[-3]
    example["election"] = path.parts[-2]
    return example

dataset_complet = dataset_complet.map(add_id, with_indices=True)



Resolving data files:   0%|          | 0/12498 [00:00<?, ?it/s]

In [18]:
import pandas as pd
metadata = pd.read_csv("data/archelect_search.csv")

In [19]:
metadata.columns

Index(['id', 'date', 'subject', 'title', 'contexte-election', 'contexte-tour',
       'cote', 'departement', 'departement-nom', 'departement-insee',
       'identifiant de circonscription', 'images', 'pdf', 'ocr_url',
       'titulaire-nom', 'titulaire-prenom', 'titulaire-sexe', 'titulaire-age',
       'titulaire-age-calcule', 'titulaire-age-tranche',
       'titulaire-profession', 'titulaire-mandat-en-cours',
       'titulaire-mandat-passe', 'titulaire-associations',
       'titulaire-autres-statuts', 'titulaire-soutien', 'titulaire-liste',
       'titulaire-decorations', 'suppleant-nom', 'suppleant-prenom',
       'suppleant-sexe', 'suppleant-age', 'suppleant-age-calcule',
       'suppleant-age-tranche', 'suppleant-profession',
       'suppleant-mandat-en-cours', 'suppleant-mandat-passe',
       'suppleant-associations', 'suppleant-autres-statuts',
       'suppleant-soutien', 'suppleant-liste', 'suppleant-decorations'],
      dtype='str')

In [20]:
metadata['titulaire-sexe'].value_counts(normalize=True)*100

titulaire-sexe
homme            83.725396
femme            10.449672
non déterminé     5.824932
Name: proportion, dtype: float64

In [21]:
metadata['contexte-tour'].value_counts(normalize=True)*100

contexte-tour
1    80.220835
2    19.779165
Name: proportion, dtype: float64

In [22]:
metadata['titulaire-profession'].value_counts(normalize=True)*100

titulaire-profession
non mentionné                                                      50.792127
professeur                                                          2.496399
avocat                                                              1.936310
chef d'entreprise                                                   1.520243
ingénieur                                                           1.288206
                                                                     ...    
chef de sociétés                                                    0.008001
professeur économie;principal-adjoint;proviseur                     0.008001
directeur  bureau d'études du bâtiment;créateur gérant sociétés     0.008001
colonel gendarmerie;consultant international                        0.008001
chef d'entreprise;directeur magazine                                0.008001
Name: proportion, Length: 2032, dtype: float64

In [23]:
metadata['titulaire-soutien'].value_counts(normalize=True)*100

titulaire-soutien
non mentionné                                                                                                                         24.291887
Parti communiste français                                                                                                             12.257961
Front national                                                                                                                         9.865578
Parti socialiste                                                                                                                       7.913266
Rassemblement pour la République;Union pour la démocratie française                                                                    6.072972
                                                                                                                                        ...    
Parti socialiste;Mouvement des radicaux de gauche;Parti communiste français;Mouvement des réformateurs                

In [24]:
# def add_soutien(example) : 
#     # print(metadata[metadata['id'] == example['id']]['titulaire-soutien'].values)
#     # print(example['id'])
#     metadata_dict = metadata.set_index('id').to_dict('index')
#     example['soutien'] = metadata[metadata['id'] == example['id']]['titulaire-soutien'].values[0]
#     example['prenom'] = metadata[metadata['id'] == example['id']]['titulaire-prenom'].values[0]
#     example['nom'] = metadata[metadata['id'] == example['id']]['titulaire-nom'].values[0]
#     example['profession'] = metadata[metadata['id'] == example['id']]['titulaire-profession'].values[0]
#     example['tour'] = metadata[metadata['id'] == example['id']]['contexte-tour'].values[0]
#     example['date'] = metadata[metadata['id'] == example['id']]['date'].values[0]
#     example['departement'] = metadata[metadata['id'] == example['id']]['departement-nom'].values[0]
    
#     return example

# dataset_complet = dataset_complet.map(add_soutien)

In [26]:
metadata_dict = metadata.set_index('id').to_dict('index')

In [ ]:
def add_soutien(example) : 
    # print(metadata[example['id']]['titulaire-soutien'].values)
    # print(example['id'])
    example['soutien'] = metadata_dict[example['id']]['titulaire-soutien']
    example['prenom'] = metadata_dict[example['id']]['titulaire-prenom']
    example['nom'] = metadata_dict[example['id']]['titulaire-nom']
    example['profession'] = metadata_dict[example['id']]['titulaire-profession']
    example['tour'] = metadata_dict[example['id']]['contexte-tour']
    example['date'] = metadata_dict[example['id']]['date']
    example['departement'] = metadata_dict[example['id']]['departement-nom']
    return example

dataset_complet = dataset_complet.map(add_soutien)

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

In [ ]:
# # 2. On divise ce bloc (ici : 10 % pour le test, 90 % pour l'entraînement)
# datasets_divises = dataset_complet.train_test_split(test_size=0.1, seed=42)

# # 3. On extrait nos deux sous-ensembles prêts à l'emploi !
# dataset_train = datasets_divises["train"]
# dataset_test = datasets_divises["test"]

In [36]:
metadata['departement-nom'].value_counts()

departement-nom
Paris                       664
Nord                        552
Bouches-du-Rhône            373
Seine-Saint-Denis           360
Hauts-de-Seine              344
                           ... 
Polynésie-française          14
Saint-Pierre-et-Miquelon      9
Guyane                        4
Wallis-et-Futuna              2
Mayotte                       1
Name: count, Length: 106, dtype: int64

In [37]:
metadata['departement-nom'].isna().sum()

np.int64(0)

In [30]:
metadata[metadata['titulaire-nom'] == "non mentionné"]

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-calcule,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations
2079,EL137_L_1981_06_088_01_1_PF_03,1981-06-14,Élections législatives;France;Ve République;As...,"Élections législatives de 1981, Vosges - 88, c...",législatives,1,EL137,88,Vosges,88 - Vosges,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Vosges écologie,non mentionné,non
2090,EL137_L_1981_06_088_04_1_PF_03,1981-06-14,France;Ve République;Élections législatives;As...,"Élections législatives de 1981, Vosges - 88, c...",législatives,1,EL137,88,Vosges,88 - Vosges,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Vosges écologie,non mentionné,non
2225,EL137_L_1981_06_092_06_1_PF_04,1981-06-14,France;Assemblée Nationale;Élections législati...,"Élections législatives de 1981, Hauts-de-Seine...",législatives,1,EL137,92,Hauts-de-Seine,92 - Hauts-de-Seine,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Aujourd'hui l'écologie,non
3284,EL174_L_1988_06_010_01_1_PF_06,1988-06-05,Élections législatives;Ve République;Assemblée...,"Élections législatives de 1988, Aube - 10, cir...",législatives,1,EL174,10,Aube,10 - Aube,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Parti ouvrier européen,non mentionné,non
3289,EL174_L_1988_06_010_02_1_PF_05,1988-06-05,Assemblée Nationale;France;Élections législati...,"Élections législatives de 1988, Aube - 10, cir...",législatives,1,EL174,10,Aube,10 - Aube,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Parti ouvrier européen,non mentionné,non
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11509,EL198_L_1993_03_095_07_1_PF_11,1993-03-21,Élections législatives;Assemblée Nationale;Ve ...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,1,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Nouveaux écologistes du rassemblement nature e...,non mentionné,non
11516,EL198_L_1993_03_095_08_1_PF_07,1993-03-21,Assemblée Nationale;Ve République;France;Élect...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,1,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non
11520,EL198_L_1993_03_095_08_1_PF_11,1993-03-21,Ve République;Assemblée Nationale;Élections lé...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,1,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Nouveaux écologistes du rassemblement nature e...,non mentionné,non
11526,EL198_L_1993_03_095_09_1_PF_06,1993-03-21,Élections législatives;Assemblée Nationale;Ve ...,"Élections législatives de 1993, Val-d'Oise - 9...",législatives,1,EL198,95,Val-d'Oise,95 - Val-d'Oise,...,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non


In [ ]:
#    example["prompt"] = (
#     f"Rédige une profession de foi pour {example['prenom']} {example['nom']}, "
#     f"de profession {example['profession']}, "
#     f"candidat soutenu par le parti {example['titulaire-soutien']} "
#     f"au tour {example['contexte-tour']} des élections législatives de {example['date']} "
#     f"dans le département : {example['departement-nom']}."
# )

In [ ]:
"colonnel de gendarmerie;consultant".split(';'";)

['colonnel de gendarmerie', 'consultant']

In [ ]:
def add_prompt(example) : 

    prompt = ["Rédige une profession de foi"]
    prenom = example['prenom'] 
    nom = example['nom']
    profession = example['profession']
    soutien = example['soutien']

    if prenom != "non mentionné" and nom != "non mentionné" : 
        prompt.append(f"pour {prenom} {nom},")
    else : 
        prompt.append("pour le candidat,")

    if profession != "non mentionné" : 
        liste_professions = profession.split(";")
        nb_professions = len(list_professions)
        if nb_professions > 1 : 
            prompt.append("de professions")
            for i in range in range(nb_professions-1) :
                prompt.append(f"{metier},")
            prompt.append(f"et {list_professions[-1]}")
        else : 
            prompt.append(f"de profession {profession},")
    
    if soutien != "non mentionné" : 
        prompt.append(f"soutenu par le parti {soutien}")

    prompt.append(f"au tour {example['tour']} des élections législatives de {example['date']}")
    prompt.append(f"dans le département : {example['departement']}.")
    
    example['prompt'] = " ".join(prompt)
    
    return example

In [47]:
dataset_complet = dataset_complet.map(add_prompt)
dataset_complet

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'prenom', 'nom', 'profession', 'tour', 'date', 'departement', 'prompt'],
    num_rows: 12498
})

In [10]:
dataset_complet['prompt'][0]

'Rédige une profession de foi pour un candidat soutenu par le parti : Parti socialiste unifié.'

In [56]:
import numpy as np
for i in np.random.randint(0, 12000, 10): 
    print(dataset_complet['prompt'][i])

Rédige une profession de foi pour Gérard Teyssier, de profession docteur en chirurgie dentaire, soutenu par le parti Gaulliste de progrès au tour 1 des élections législatives de 1981-06-14 dans le département : Bouches-du-Rhône.
Rédige une profession de foi pour Jean-Marie Schneider, de profession gérant société, soutenu par le parti Front national au tour 1 des élections législatives de 1993-03-21 dans le département : Haut-Rhin.
Rédige une profession de foi pour Régis Barailla, au tour 1 des élections législatives de 1988-06-05 dans le département : Aude.
Rédige une profession de foi pour le candidat, soutenu par le parti Nouveaux écologistes du rassemblement nature et animaux au tour 1 des élections législatives de 1993-03-21 dans le département : Vaucluse.
Rédige une profession de foi pour Marie-Annick Trentarossi, au tour 2 des élections législatives de 1993-03-28 dans le département : Yvelines.
Rédige une profession de foi pour Jean-Louis Mas, de profession rédacteur tarificateur

In [12]:
# Data cleaning 

import re

def reparer_mots_coupes(texte):
    # RÈGLE 1 : Les tirets de fin de ligne (ex: "consolida-\ntion")
    # On cherche : (Des lettres) + un tiret + (des sauts de ligne ou espaces) + (Des lettres)
    # Le r'\1\2' dit à Python : recolle le groupe 1 (consolida) et le groupe 2 (tion) sans le tiret.
    # Note : Le [a-zA-ZÀ-ÿ] permet d'inclure les accents français (é, à, ç...)
    texte = re.sub(r'([a-zA-ZÀ-ÿ]+)-\s*\n\s*([a-zA-ZÀ-ÿ]+)', r'\1\2', texte)
    
    # RÈGLE 2 : Les tirets suivis d'un espace (ex: "plura- lisme")
    # On cherche : (Des lettres) + un tiret + (un ou plusieurs espaces) + (Des lettres)
    texte = re.sub(r'([a-zA-ZÀ-ÿ]+)-\s+([a-zA-ZÀ-ÿ]+)', r'\1\2', texte)
    
    return texte

# --- TEST POUR VÉRIFIER ---
texte_brut = """
maire-adjointe à Bourg-en-bresse
une étape importante pour la consolida-
tion de cette victoire.
le respect du plura- lisme politique.
- vive la France 
- et vive ce qu'il en reste
"""

print("AVANT :")
print(texte_brut)

print("APRÈS :")
print(reparer_mots_coupes(texte_brut))

AVANT :

maire-adjointe à Bourg-en-bresse
une étape importante pour la consolida-
tion de cette victoire.
le respect du plura- lisme politique.
- vive la France 
- et vive ce qu'il en reste

APRÈS :

maire-adjointe à Bourg-en-bresse
une étape importante pour la consolidation de cette victoire.
le respect du pluralisme politique.
- vive la France 
- et vive ce qu'il en reste



In [110]:
def add_messages(example) : 
    example['messages'] = [{"role" : "user", "content" : f"{example['prompt']}"},
                           {"role" : "assistant", "content" : f"{example['text']}"}]
    return example

In [111]:
dataset_complet = dataset_complet.map(add_messages)

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

In [113]:
dataset_complet['messages']

Column([[{'content': 'Rédige une profession de foi pour un candidat soutenu par le parti : Parti socialiste unifié.', 'role': 'user'}, {'content': "ELECTIONS LEGISLATIVES - 14 JUIN 1981 AIN 1e CIRCONSCRIPTION\nMicheline ANTONUCCI\nPOUR QUE ÇA DURE ...\nLa victoire de François Mitterrand, le 10 mai, a exprimé la volonté de changement de la grande majorité des travailleurs, des hommes et des femmes de ce pays.\nLes élections législatives sont une étape importante pour la consolida- tion de cette victoire : il faut une majorité de gauche à l'Assemblée Nationale.\nmaire-adjointe à Bourg-en-bresse assistante sociale\n... IL FAUT QUE TOUTES LES FORCES DE GAUCHE ...\nUne condition essentielle de la victoire est la reconnaissance du plura- lisme respectant les différences des forces politiques de gauche.\nCela est nécessaire pour éviter des alliances centristes qui réduiraient à néant la victoire du 10 mai.\nDans cet esprit, le P.S.U. entend au premier tour de ces élections met- tre l'accent s

In [114]:
def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""
    # Format answers
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

In [115]:
dataset_complet = dataset_complet.map(format_prompt)

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

In [116]:
dataset_complet['text'][0]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 14 Mar 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nRédige une profession de foi pour un candidat soutenu par le parti : Parti socialiste unifié.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nELECTIONS LEGISLATIVES - 14 JUIN 1981 AIN 1e CIRCONSCRIPTION\nMicheline ANTONUCCI\nPOUR QUE ÇA DURE ...\nLa victoire de François Mitterrand, le 10 mai, a exprimé la volonté de changement de la grande majorité des travailleurs, des hommes et des femmes de ce pays.\nLes élections législatives sont une étape importante pour la consolida- tion de cette victoire : il faut une majorité de gauche à l'Assemblée Nationale.\nmaire-adjointe à Bourg-en-bresse assistante sociale\n... IL FAUT QUE TOUTES LES FORCES DE GAUCHE ...\nUne condition essentielle de la victoire est la reconnaissance du plura- lisme respectant les différences des forces politiques de gauche.\nCel

In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# # 4-bit quantization configuration - Q in QLoRA
# bnb_config = BitsAndBytesConfig(
# load_in_4bit=True, # Use 4-bit precision model loading
# bnb_4bit_quant_type="nf4", # Quantization type
# bnb_4bit_compute_dtype="float16", # Compute dtype
# bnb_4bit_use_double_quant=True, # Apply nested quantization
# )
# # Load the model to train on the GPU
# model = AutoModelForCausalLM.from_pretrained(
# model_name,
# device_map="auto",
# # Leave this out for regular SFT
# quantization_config=bnb_config,
# )
# model.config.use_cache = False
# model.config.pretraining_tp = 1
# # Load LLaMA tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# tokenizer.pad_token = "<PAD>"
# tokenizer.padding_side = "left"

In [118]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 1. Chargement et configuration du Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# On recycle le jeton de fin existant au lieu d'en inventer un
tokenizer.pad_token = tokenizer.eos_token 
tokenizer.padding_side = "left" # Pour l'entraînement (Causal LM), on ajoute le remplissage à la fin

# 2. Chargement du Modèle (optimisé pour Apple Silicon)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="mps",           # On envoie directement sur la puce graphique de votre Mac
    torch_dtype=torch.bfloat16  # Précision 16-bit : le modèle pèsera environ 2.5 Go en RAM
)
model.config.use_cache = False
model.config.pretraining_tp = 1

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [119]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
# Prepare LoRA Configuration
peft_config = LoraConfig(
lora_alpha=32, # LoRA Scaling
lora_dropout=0.1, # Dropout for LoRA Layers
r=64, # Rank
bias="none",
task_type="CAUSAL_LM",
target_modules= # Layers to target
["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj",
"down_proj"]
)
# Prepare model for training
#model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [121]:
from transformers import TrainingArguments
output_dir = "./results"
# Training arguments
training_arguments = TrainingArguments(
output_dir=output_dir,
per_device_train_batch_size=2,
gradient_accumulation_steps=4,
#optim="paged_adamw_32bit",
optim = "adamw_torch",
learning_rate=2e-4,
lr_scheduler_type="cosine",
num_train_epochs=1,
logging_steps=10,
#fp16=True,
bf16=True,
gradient_checkpointing=True
)

In [128]:
dataset_light = dataset_complet.remove_columns(['prompt'])

In [129]:
dataset_light

Dataset({
    features: ['text', 'id', 'annee', 'election', 'soutien', 'messages'],
    num_rows: 12498
})

In [131]:
from trl import SFTTrainer, SFTConfig
# Set supervised fine-tuning parameters
training_arguments = SFTConfig(
output_dir=output_dir,
per_device_train_batch_size=2,
gradient_accumulation_steps=4,
#optim="paged_adamw_32bit",
optim = "adamw_torch",
learning_rate=2e-4,
lr_scheduler_type="cosine",
num_train_epochs=1,
logging_steps=10,
#fp16=True,
bf16=True,
gradient_checkpointing=True,
dataset_text_field="text",
max_length=512
)
trainer = SFTTrainer(
model=model,
train_dataset=dataset_light,
processing_class=tokenizer,
args=training_arguments,
# Leave this out for regular SFT
#peft_config=peft_config,
)
# Train model
trainer.train()
# Save QLoRA weights
trainer.model.save_pretrained("Llama-1B-Archelec-LoRA")

Tokenizing train dataset:   0%|          | 0/12498 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12498 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


KeyboardInterrupt: 